In [1]:
# Then import McNemar test
import pathlib
import pandas as pd
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar

In [11]:
model_name = 'Qwen2'
is_cot = False
socio_demographic_variables_str = 'gender_race_political'
results_base_socdem = pathlib.Path( 'predictions_dem') / f"{model_name}" / 'cleaned'
results_base_baseline = pathlib.Path('predictions')  / 'cleaned'
filename_baseline = f'cleaned_predictions_{model_name}_{"CoT" if is_cot else "noCoT"}_baseline.csv'
filename_socdem = f'cleaned_predictions_{model_name}_{"CoT" if is_cot else "noCoT"}_{socio_demographic_variables_str}.csv'

df_baseline = pd.read_csv(results_base_baseline / filename_baseline)
df_intersection = pd.read_csv(results_base_socdem / filename_socdem)

In [12]:
merged_df = df_intersection.merge(df_baseline,
                                 on=['postId', 'annId', 'offensiveYN'], suffixes=('', '_baseline'))

In [13]:
merged_df = merged_df[['offensiveYN', 'prediction', 'prediction_baseline']]

In [14]:
merged_df_positive = merged_df[merged_df['offensiveYN'] == 1]

In [ ]:
def extract_correct(df, correct_column='offensiveYN', prediction_column='prediction'):
    """
    Extracts the correct predictions from the DataFrame as a list of 1s and 0s with 1 indicating a correct prediction.
    """
    correct = [1 if row[prediction_column] == row[correct_column] else 0 for _, row in df.iterrows()]
    return correct

def get_mcnemar_results(correct_intersectional, correct_baseline, option='greater'):
    contingency_table = pd.crosstab(correct_intersectional, correct_baseline, rownames=['Intersectional'], colnames=['Baseline'])
    result = mcnemar(contingency_table, exact=True)
    print("McNemar's Test Result:")
    if option == 'two-sided':
        print(f"Statistic: {result.statistic}, p-value: {result.pvalue}")
    elif option == 'greater':
        # To use one-sided test, we need to check the contingency table
        if contingency_table[0][1] < contingency_table[1][0]:
            raise ValueError("The contingency table does not meet the requirements for a one-sided test.")
        print(f"Statistic: {result.statistic}, p-value (greater): {result.pvalue / 2}")

In [37]:
correct_intersection = extract_correct(df_intersection)
correct_baseline = extract_correct(df_baseline)
get_mcnemar_results(correct_intersection, correct_baseline, option='greater')

McNemar's Test Result:


ValueError: The contingency table does not meet the requirements for a one-sided test.